In [35]:
import pandas as pd

# Read and display sample_submission.csv
sample_submission_df = pd.read_csv('/content/sample_submission.csv')
# Read and display /content/test.csv
test_df = pd.read_csv('/content/test.csv')
# Read and display /content/train.csv
train_df = pd.read_csv('/content/train.csv')


In [36]:
sample_submission_df.head()

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A B C
3,4,A B C
4,5,A B C


In [37]:
test_df.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [38]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [39]:
# Section 1: Install & Import Libraries
!pip install gensim

import re
import string
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import Word2Vec

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [40]:
# Section 2: Text Cleaning, Tokenization & Missing Data Handling

def clean_text(text):

    # Handle missing values
    if not isinstance(text, str):
        text = ''

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = re.sub(
        f'[{re.escape(string.punctuation)}]',
        '',
        text
    )

    # Tokenization
    tokens = word_tokenize(text)

    # Stopword removal
    stop_words = set(stopwords.words('english'))

    filtered_tokens = [
        word
        for word in tokens
        if word not in stop_words and word.isalpha()
    ]

    return ' '.join(filtered_tokens), filtered_tokens

In [41]:
# Section 3: Apply Cleaning to Train and Test Data
text_cols = ['prompt', 'A', 'B', 'C', 'D', 'E']

print("Cleaning text...")

for df in [train_df, test_df]:

    for col in text_cols:

        df[col] = df[col].fillna('')

        df[f'{col}_cleaned'], df[f'{col}_tokens'] = zip(
            *df[col].apply(clean_text)
        )

print("Cleaning completed.")

Cleaning text...
Cleaning completed.


In [42]:
# Section 4: Generate TF-IDF Embeddings

print("Generating TF-IDF embeddings...")

all_cleaned_text = []

for col in [f'{c}_cleaned' for c in text_cols]:

    all_cleaned_text.extend(
        train_df[col].tolist()
    )

# Fit only on training data
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000
)

tfidf_vectorizer.fit(all_cleaned_text)

# Train data embeddings
train_df['prompt_tfidf'] = list(
    tfidf_vectorizer.transform(
        train_df['prompt_cleaned']
    )
)

for col in ['A', 'B', 'C', 'D', 'E']:

    train_df[f'{col}_tfidf'] = list(
        tfidf_vectorizer.transform(
            train_df[f'{col}_cleaned']
        )
    )

# Test data embeddings
test_df['prompt_tfidf'] = list(
    tfidf_vectorizer.transform(
        test_df['prompt_cleaned']
    )
)

for col in ['A', 'B', 'C', 'D', 'E']:

    test_df[f'{col}_tfidf'] = list(
        tfidf_vectorizer.transform(
            test_df[f'{col}_cleaned']
        )
    )

print("TF-IDF completed.")

Generating TF-IDF embeddings...
TF-IDF completed.


In [43]:
# Section 5: Generate Word2Vec Embeddings

print("Training Word2Vec model...")

all_tokens = []

for col in [f'{c}_tokens' for c in text_cols]:

    all_tokens.extend(
        train_df[col].tolist()
    )

word2vec_model = Word2Vec(
    sentences=all_tokens,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

print("Word2Vec training completed.")

Training Word2Vec model...
Word2Vec training completed.


In [44]:
# Section 6: Convert Sentences to Word2Vec Embeddings
def get_word2vec_embedding(
    tokens,
    model,
    vector_size=100
):

    vectors = []

    for token in tokens:

        if token in model.wv:
            vectors.append(
                model.wv[token]
            )

    if len(vectors) > 0:

        return np.mean(
            vectors,
            axis=0
        )

    return np.zeros(vector_size)

In [45]:
# Section 7: Apply Word2Vec Embeddings
train_df['prompt_w2v'] = train_df[
    'prompt_tokens'
].apply(
    lambda x:
    get_word2vec_embedding(
        x,
        word2vec_model
    )
)

for col in ['A', 'B', 'C', 'D', 'E']:

    train_df[f'{col}_w2v'] = train_df[
        f'{col}_tokens'
    ].apply(
        lambda x:
        get_word2vec_embedding(
            x,
            word2vec_model
        )
    )

test_df['prompt_w2v'] = test_df[
    'prompt_tokens'
].apply(
    lambda x:
    get_word2vec_embedding(
        x,
        word2vec_model
    )
)

for col in ['A', 'B', 'C', 'D', 'E']:

    test_df[f'{col}_w2v'] = test_df[
        f'{col}_tokens'
    ].apply(
        lambda x:
        get_word2vec_embedding(
            x,
            word2vec_model
        )
    )

print("Word2Vec embeddings generated.")

Word2Vec embeddings generated.


In [46]:
# Section 8: Cosine Similarity Function
def compute_cosine_similarity(
    vec1,
    vec2
):

    # TF-IDF sparse matrices
    if hasattr(vec1, "toarray"):

        return cosine_similarity(
            vec1,
            vec2
        )[0][0]

    # Word2Vec vectors
    return cosine_similarity(
        vec1.reshape(1, -1),
        vec2.reshape(1, -1)
    )[0][0]

In [47]:
# Section 9: Example Similarity Calculation
print("\nExample Similarities")

tfidf_similarity = compute_cosine_similarity(
    train_df['prompt_tfidf'].iloc[0],
    train_df['A_tfidf'].iloc[0]
)

print(
    f"TF-IDF Similarity: {tfidf_similarity:.4f}"
)

w2v_similarity = compute_cosine_similarity(
    train_df['prompt_w2v'].iloc[0],
    train_df['A_w2v'].iloc[0]
)

print(
    f"Word2Vec Similarity: {w2v_similarity:.4f}"
)


Example Similarities
TF-IDF Similarity: 0.1596
Word2Vec Similarity: 0.7747


In [48]:
# Section 10: AP@3 Calculation
def calculate_ap_at_k(
    actual,
    predicted,
    k=3
):

    predicted = predicted[:k]

    for i, p in enumerate(predicted):

        if p == actual:

            return 1.0 / (i + 1)

    return 0.0

In [49]:
# Section 11: mAP@3 Calculation
def calculate_map_k(
    df,
    embedding_type='tfidf',
    k=3
):

    options = [
        'A',
        'B',
        'C',
        'D',
        'E'
    ]

    ap_scores = []

    for _, row in df.iterrows():

        prompt_vec = row[
            f'prompt_{embedding_type}'
        ]

        actual_answer = row['answer']

        similarities = []

        for opt in options:

            option_vec = row[
                f'{opt}_{embedding_type}'
            ]

            similarity = compute_cosine_similarity(
                prompt_vec,
                option_vec
            )

            similarities.append(
                (
                    opt,
                    similarity
                )
            )

        similarities.sort(
            key=lambda x: x[1],
            reverse=True
        )

        predicted_labels = [
            opt
            for opt, sim
            in similarities
        ]

        ap = calculate_ap_at_k(
            actual_answer,
            predicted_labels,
            k
        )

        ap_scores.append(ap)

    return np.mean(ap_scores)

In [50]:
# Section 12: Calculate Final mAP@3 Scores
print("\nCalculating mAP@3")

map3_tfidf = calculate_map_k(
    train_df,
    embedding_type='tfidf',
    k=3
)

print(
    f"TF-IDF mAP@3: {map3_tfidf:.4f}"
)

map3_w2v = calculate_map_k(
    train_df,
    embedding_type='w2v',
    k=3
)

print(
    f"Word2Vec mAP@3: {map3_w2v:.4f}"
)


Calculating mAP@3
TF-IDF mAP@3: 0.2871
Word2Vec mAP@3: 0.3830


In [51]:
# Section 13: Concept Summary
print("""
NLP Tasks Completed:

1. Text Cleaning
   - Lowercasing
   - Punctuation Removal
   - Stopword Removal

2. Tokenization
   - Split text into words

3. Missing Data Handling
   - Replaced NaN with empty strings

4. TF-IDF Embeddings
   - Frequency-based representation

5. Word2Vec Embeddings
   - Semantic representation

6. Cosine Similarity
   - Measures similarity between prompt and options

7. mAP@3
   - Evaluates ranking quality
   - Correct answer at:
       Rank 1 -> 1.0
       Rank 2 -> 0.5
       Rank 3 -> 0.333
       Outside Top 3 -> 0
""")


NLP Tasks Completed:

1. Text Cleaning
   - Lowercasing
   - Punctuation Removal
   - Stopword Removal

2. Tokenization
   - Split text into words

3. Missing Data Handling
   - Replaced NaN with empty strings

4. TF-IDF Embeddings
   - Frequency-based representation

5. Word2Vec Embeddings
   - Semantic representation

6. Cosine Similarity
   - Measures similarity between prompt and options

7. mAP@3
   - Evaluates ranking quality
   - Correct answer at:
       Rank 1 -> 1.0
       Rank 2 -> 0.5
       Rank 3 -> 0.333
       Outside Top 3 -> 0



In [52]:
# Count frequency of each answer option
answer_counts = train_df['answer'].value_counts()

print("Frequency Distribution:")
print(answer_counts)

# Most frequent count
most_frequent = answer_counts.max()

# Least frequent count
least_frequent = answer_counts.min()

# Required sum
result = most_frequent + least_frequent

print("\nMost Frequent Count:", most_frequent)
print("Least Frequent Count:", least_frequent)
print("Sum:", result)

Frequency Distribution:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Most Frequent Count: 490
Least Frequent Count: 324
Sum: 814


In [53]:
import string

# Create a translator to remove punctuation
translator = str.maketrans('', '', string.punctuation)

vocabulary = set()

for text in train_df['prompt'].fillna(''):

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(translator)

    # Split by whitespace
    words = text.split()

    # Add words to vocabulary
    vocabulary.update(words)

# Total unique words
vocab_size = len(vocabulary)

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 859


In [54]:
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Get prompt from Row ID 1
prompt = train_df.loc[train_df['id'] == 1, 'prompt'].iloc[0]

# Convert to lowercase
prompt = prompt.lower()

# Remove punctuation
prompt = prompt.translate(
    str.maketrans('', '', string.punctuation)
)

# Split into words
words = prompt.split()

# Remove stop words
filtered_words = [
    word
    for word in words
    if word not in ENGLISH_STOP_WORDS
]

# Number of words remaining
print("Words left:", len(filtered_words))

# Optional: view remaining words
print(filtered_words)

Words left: 13
['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


In [55]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine prompt and all options into one text per row
combined_text = (
    train_df['prompt'].fillna('') + ' ' +
    train_df['A'].fillna('') + ' ' +
    train_df['B'].fillna('') + ' ' +
    train_df['C'].fillna('') + ' ' +
    train_df['D'].fillna('') + ' ' +
    train_df['E'].fillna('')
)

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english')

# Fit on the combined text
vectorizer.fit(combined_text)

# Vocabulary size
vocab_size = len(vectorizer.get_feature_names_out())

print("Vocabulary Size:", vocab_size)

Vocabulary Size: 2762


In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create combined text and fit vectorizer (same as Question 3)
combined_text = (
    train_df['prompt'].fillna('') + ' ' +
    train_df['A'].fillna('') + ' ' +
    train_df['B'].fillna('') + ' ' +
    train_df['C'].fillna('') + ' ' +
    train_df['D'].fillna('') + ' ' +
    train_df['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

# Get Row ID 1
row = train_df.loc[train_df['id'] == 1].iloc[0]

prompt = row['prompt']
option_a = row['A']

# Transform to TF-IDF vectors
prompt_vec = vectorizer.transform([prompt])
option_a_vec = vectorizer.transform([option_a])

# Compute cosine similarity
similarity = cosine_similarity(
    prompt_vec,
    option_a_vec
)[0][0]

print("Cosine Similarity:", round(similarity, 4))

Cosine Similarity: 0.272


In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine all text for TF-IDF fitting
combined_text = (
    train_df['prompt'].fillna('') + ' ' +
    train_df['A'].fillna('') + ' ' +
    train_df['B'].fillna('') + ' ' +
    train_df['C'].fillna('') + ' ' +
    train_df['D'].fillna('') + ' ' +
    train_df['E'].fillna('')
)

# Fit vectorizer
vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

options = ['A', 'B', 'C', 'D', 'E']

correct_predictions = 0
total_rows = len(train_df)

for _, row in train_df.iterrows():

    # TF-IDF vector for prompt
    prompt_vec = vectorizer.transform([str(row['prompt'])])

    similarities = {}

    for opt in options:

        option_text = str(row[opt])

        option_vec = vectorizer.transform([option_text])

        sim = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities[opt] = sim

    # Option with highest similarity
    predicted_answer = max(
        similarities,
        key=similarities.get
    )

    # Compare with actual answer
    if predicted_answer == row['answer']:
        correct_predictions += 1

# Percentage accuracy
accuracy_percentage = (
    correct_predictions / total_rows
) * 100

print(f"Correct Predictions: {correct_predictions}")
print(f"Total Rows: {total_rows}")
print(f"Percentage Accuracy: {accuracy_percentage:.2f}%")

Correct Predictions: 271
Total Rows: 2000
Percentage Accuracy: 13.55%


In [58]:
actual = 'C'
predicted = ['C', 'A', 'B']

def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)

    return 0.0

score = apk(actual, predicted, k=3)

print(score)

1.0


In [59]:
actual = 'B'
predicted = ['D', 'B', 'E']

def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)

    return 0.0

score = apk(actual, predicted, k=3)

print(score)

0.5


In [60]:
import numpy as np

# Frequency distribution of answers
answer_counts = train_df['answer'].value_counts()

print("Answer Frequencies:")
print(answer_counts)

# Top 3 most frequent answers
top3_answers = answer_counts.index[:3].tolist()

print("\nStatic Predictions:")
print(top3_answers)

# AP@3 function
def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)

    return 0.0

# Calculate AP@3 for every row
scores = []

for actual in train_df['answer']:
    scores.append(
        apk(actual, top3_answers, k=3)
    )

# MAP@3
map3 = np.mean(scores)

print(f"\nMajority Class Baseline MAP@3 = {map3:.6f}")

Answer Frequencies:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Static Predictions:
['B', 'C', 'A']

Majority Class Baseline MAP@3 = 0.421250


In [61]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------------------
# Step 1: Fit TF-IDF Vectorizer
# ----------------------------------

combined_text = (
    train_df['prompt'].fillna('') + ' ' +
    train_df['A'].fillna('') + ' ' +
    train_df['B'].fillna('') + ' ' +
    train_df['C'].fillna('') + ' ' +
    train_df['D'].fillna('') + ' ' +
    train_df['E'].fillna('')
)

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text)

# ----------------------------------
# Step 2: AP@3 Function
# ----------------------------------

def apk(actual, predicted, k=3):

    predicted = predicted[:k]

    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)

    return 0.0

# ----------------------------------
# Step 3: Evaluate Every Row
# ----------------------------------

options = ['A', 'B', 'C', 'D', 'E']

scores = []

for _, row in train_df.iterrows():

    prompt_vec = vectorizer.transform(
        [str(row['prompt'])]
    )

    similarities = []

    for option in options:

        option_vec = vectorizer.transform(
            [str(row[option])]
        )

        similarity = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

        similarities.append(
            (option, similarity)
        )

    # Sort by similarity descending
    similarities.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # Top 3 predictions
    top3_predictions = [
        option
        for option, score
        in similarities[:3]
    ]

    # Calculate AP@3
    score = apk(
        row['answer'],
        top3_predictions,
        k=3
    )

    scores.append(score)

# ----------------------------------
# Step 4: Final MAP@3
# ----------------------------------

map3 = np.mean(scores)

print(f"TF-IDF Pipeline MAP@3 = {map3:.6f}")

TF-IDF Pipeline MAP@3 = 0.296167
